# Mitandao ya neva ya kurudiarudia

Katika moduli iliyopita, tulijadili uwakilishi mzito wa semantiki wa maandishi. Mimarishaji ambayo tumekuwa tukitumia hugusa maana iliyojumlishwa ya maneno katika sentensi, lakini haizingi tarehe ya maneno, kwa sababu operesheni ya kujumuisha inayofuata embeddings huondoa taarifa hii kutoka kwenye maandishi asilia. Kwa sababu mifano hii haiwezi kuwakilisha mpangilio wa maneno, haiwezi kutatua kazi ngumu zaidi au zenye utata kama vile uundaji wa maandishi au kujibu maswali.

Ili kushika maana ya mlolongo wa maandishi, tutatumia usanifu wa mtandao wa neva unaoitwa **mtandao wa neva wa kurudiarudia**, au RNN. Tunapotumia RNN, tunapitisha sentensi yetu kupitia mtandao neno moja kwa wakati, na mtandao hutengeneza aina fulani ya **hali**, ambayo tunapitisha tena kwa mtandao pamoja na neno linalofuata.

![Picha inaonyesha mfano wa uzalishaji wa mtandao wa neva wa kurudiarudia.](../../../../../translated_images/sw/rnn.27f5c29c53d727b5.webp)

Kwa kuzingatia mlolongo wa pembejeo wa tokeni $X_0,\dots,X_n$, RNN huunda mlolongo wa vitengo vya mtandao wa neva, na huandaa mlolongo huu mwishoni-mwisho kwa kutumia backpropagation. Kila kitengo cha mtandao kinachukua jozi $(X_i,S_i)$ kama pembejeo, na hutengeneza $S_{i+1}$ kama matokeo. Hali ya mwisho $S_n$ au pato $Y_n$ huingizwa katika kieweka daraja la mstari kutoa matokeo. Vitengo vyote vya mtandao vinashiriki uzito ule ule, na vinafundishwa mwishoni-mwisho kwa kutumia mara moja ya kurudisha nyuma (back propagation).

> Mchoro ulio juu unaonyesha mtandao wa neva wa kurudiarudia ukiwa kwenye fomu iliyofunguliwa (kushoto), na katika uwakilishi mfupi wa kurudiarudia (kulia). Ni muhimu kutambua kwamba Seli zote za RNN zina **uzito wa kushirikiana** ule ule.

Kwa sababu vekta za hali $S_0,\dots,S_n$ hupitishwa kupitia mtandao, RNN ina uwezo wa kujifunza utegemezi wa mfululizo kati ya maneno. Kwa mfano, wakati neno *not* linapotokea mahali fulani katika mlolongo, inaweza kujifunza kukanusha vipengele fulani ndani ya vekta ya hali.

Ndani, kila seli ya RNN ina matriki mbili za uzito: $W_H$ na $W_I$, na upendeleo $b$. Kila hatua ya RNN, ikipewa pembejeo $X_i$ na hali ya pembejeo $S_i$, hali ya pato huhesabiwa kama $S_{i+1} = f(W_H\times S_i + W_I\times X_i+b)$, ambapo $f$ ni kazi ya uanzishwaji (kawaida $\tanh$).

> Kwa matatizo kama uundaji wa maandishi (yatakayojadiliwa katika kitengo kinachofuata) au tafsiri ya mashine pia tunahitaji kupata thamani ya pato kila hatua ya RNN. Katika kesi hii, kuna matriki nyingine $W_O$, na pato huhesabiwa kama $Y_i=f(W_O\times S_i+b_O)$.

Tuwe tione jinsi mitandao ya neva ya kurudiarudia inaweza kutusaidia kuainisha seti yetu ya habari.

> Kwa mazingira ya sandbox, tunahitaji kuendesha seli ifuatayo kuhakikisha maktaba inayohitajika imewekwa, na data imetanguliwa kupakuliwa. Ikiwa unafanya kazi kwa ndani, unaweza kuruka seli ifuatayo.


In [1]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

Wakati wa kufundisha mifano mikubwa, ugawaji wa kumbukumbu ya GPU unaweza kuwa tatizo. Pia tunaweza kuhitaji kujaribu ukubwa tofauti wa minibatch, ili data ifae kwenye kumbukumbu ya GPU yetu, lakini mafundisho yafanye kazi kwa kasi ya kutosha. Ikiwa unafanya kazi na msimbo huu kwenye mashine yako mwenyewe ya GPU, unaweza kujaribu kurekebisha ukubwa wa minibatch ili kuharakisha mafunzo.

> **Note**: Matoleo fulani ya madereva ya NVidia yanajulikana kutoruhusu kumbukumbu kuachiwa baada ya kufundisha mfano. Tunaendesha mifano kadhaa katika daftari hili, na inaweza kusababisha kumbukumbu kutumika kwa kiasi kikubwa katika mipangilio fulani, hasa ikiwa unafanya majaribio yako kama sehemu ya daftari hili. Ikiwa utakutana na makosa ya ajabu unapoanza kufundisha mfano, unaweza kutaka kuwasha upya kernel ya daftari.


In [3]:
batch_size = 16
embed_size = 64

## Mwekuzaji RNN Rahisi

Katika kesi ya RNN rahisi, kila kitengo cha kurudiwa ni mtandao rahisi wa mstari, ambao hupokea vector ya ingizo na vector ya hali, na hutengeneza vector mpya ya hali. Katika Keras, hii inaweza kuwakilishwa na safu ya `SimpleRNN`.

Ingawa tunaweza kupitisha tokeni zilizoambatanishwa moja kwa moja kwa safu ya RNN, hii si wazo zuri kwa sababu ya wingi wao mkubwa wa vipimo. Kwa hiyo, tutatumia safu ya kuingiza ili kupunguza wingi wa vipimo vya vector za maneno, ikifuatiwa na safu ya RNN, na mwisho mwekuzaji wa `Dense`.

> **Kumbuka**: Katika kesi ambapo wingi wa vipimo si mkubwa sana, kwa mfano wakati wa kutumia uugaji tokeni za kiwango cha herufi, inaweza kuwa na maana kupitisha tokeni zilizoambatanishwa moja kwa moja ndani ya seli ya RNN.


In [4]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
text_vectorization (TextVect (None, None)              0         
_________________________________________________________________
embedding (Embedding)        (None, None, 64)          1280000   
_________________________________________________________________
simple_rnn (SimpleRNN)       (None, 16)                1296      
_________________________________________________________________
dense (Dense)                (None, 4)                 68        
Total params: 1,281,364
Trainable params: 1,281,364
Non-trainable params: 0
_________________________________________________________________


> **Kumbuka:** Tunatumia safu ya embedding isiyo na mafunzo hapa kwa ajili ya urahisi, lakini kwa matokeo bora tunaweza kutumia safu ya embedding iliyo na mafunzo kwa kutumia Word2Vec, kama ilivyoelezwa katika kipengele kilichopita. Itakuwa zoezi zuri kwako kubadilisha msimbo huu ufanye kazi na embeddings zilizo na mafunzo.

Sasa hebu tufunze RNN yetu. RNN kwa ujumla ni ngumu kufundisha, kwa sababu mara tu seli za RNN zinapopengezwa kufuatia mfuatano wa urefu wake, idadi ya safu zinazohusika katika backpropagation huwa kubwa sana. Hivyo tunahitaji kuchagua kiwango kidogo cha kujifunza, na kufundisha mtandao kwa seti kubwa ya data kutengeneza matokeo bora. Hii inaweza kuchukua muda mrefu, kwa hiyo kutumia GPU ni bora.

Ili kuharakisha mambo, tutafundisha mfano wa RNN tu kwenye vichwa vya habari, tukiacha maelezo. Unaweza kujaribu kufundisha na maelezo na kuona kama unaweza kufanya mfano ufundishwe.


In [5]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

Training vectorizer


In [6]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

7500/7500 [==============================] - 82s 11ms/step - loss: 0.6629 - acc: 0.7623 - val_loss: 0.5559 - val_acc: 0.7995


> **Kumbuka** kuwa usahihi huenda ukawa mdogo hapa, kwa sababu tunafunza tu kwa vichwa vya habari.


## Revisiting variable sequences 

Remember that the `TextVectorization` layer will automatically pad sequences of variable length in a minibatch with pad tokens. It turns out that those tokens also take part in training, and they can complicate convergence of the model.

There are several approaches we can take to minimize the amount of padding. One of them is to reorder the dataset by sequence length and group all sequences by size. This can be done using the `tf.data.experimental.bucket_by_sequence_length` function (see [documentation](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)). 

Another approach is to use **masking**. In Keras, some layers support additional input that shows which tokens should be taken into account when training. To incorporate masking into our model, we can either include a separate `Masking` layer ([docs](https://keras.io/api/layers/core_layers/masking/)), or we can specify the `mask_zero=True` parameter of our `Embedding` layer.

> **Note**: This training will take around 5 minutes to complete one epoch on the whole dataset. Feel free to interrupt training at any time if you run out of patience. What you can also do is limit the amount of data used for training, by adding `.take(...)` clause after `ds_train` and `ds_test` datasets.


In [7]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

7500/7500 [==============================] - 371s 49ms/step - loss: 0.5401 - acc: 0.8079 - val_loss: 0.3780 - val_acc: 0.8822


Sasa tunapoitumia masking, tunaweza kufundisha modeli kwenye seti yote ya data ya vichwa na maelezo.

> **Kumbuka**: Je, umeona kuwa tumekuwa tukitumia vectorizer iliyo funzwa kwenye vichwa vya habari, na si mwili mzima wa makala? Inawezekana, hii inaweza kusababisha baadhi ya tokeni kupuuzwa, hivyo ni bora kufanya upya mafunzo ya vectorizer. Hata hivyo, inaweza kuwa na athari ndogo sana, kwa hivyo tutaendelea kutumia vectorizer iliyofunzwa awali kwa ajili ya urahisi.


## LSTM: Kumbukumbu ya muda mrefu na mfupi

Moja ya matatizo makuu ya RNN ni **kupotea kwa gradients**. RNN zinaweza kuwa ndefu sana, na zinaweza kupata shida kusambaza gradients zote hadi tabaka la kwanza la mtandao wakati wa backpropagation. Inapotokea hivi, mtandao hauwezi kujifunza uhusiano kati ya tokeni za mbali. Njia moja ya kuepuka tatizo hili ni kuanzisha **usimamizi wa hali wazi** kwa kutumia **milango**. Miundo mibili maarufu inayotumia milango ni **kumbukumbu ya muda mrefu na mfupi** (LSTM) na **gated relay unit** (GRU). Tutazungumzia LSTM hapa.

![Picha inayonyesha mfano wa seli ya kumbukumbu ya muda mrefu na mfupi](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

Mtandao wa LSTM umeandaliwa kwa mtindo unaofanana na RNN, lakini kuna hali mbili zinazopitishwa kutoka tabaka moja hadi jingine: hali halisi $c$, na vector fiche $h$. Kila kitengo, vector fiche $h_{t-1}$ huunganishwa na ingizo $x_t$, na pamoja hudhibiti kinachotokea kwa hali $c_t$ na matokeo $h_{t}$ kupitia **milango**. Kila mlango una ufanyakazi wa sigmoid (matokeo ndani ya kiwango $[0,1]$), ambao unaweza kufikirika kama kinyago cha bit wakati unazidishwa na vector ya hali. LSTM zina milango ifuatayo (kutoka kushoto kwenda kulia kwenye picha juu):
* **mlango wa kusahau** ambao huamua vipengele gani vya vector $c_{t-1}$ tunapaswa kusahau, na vipengele gani kuziweka.
* **mlango wa ingizo** ambao huamua ni kiasi gani cha taarifa kutoka kwa vector ya ingizo na vector fiche ya awali kinapaswa kuingizwa katika vector ya hali.
* **mlango wa matokeo** ambao huchukua vector mpya ya hali na kuamua ni vipengele gani vitatumika kutoa vector mpya fiche $h_t$.

Vipengele vya hali $c$ vinaweza kufikiriwa kama bendera zinazoweza kuwashwa au kuzimwa. Kwa mfano, tunapokutana na jina *Alice* katika mlolongo, tunakisia kwamba linarejelea mwanamke, na tunainua bendera katika hali inayosema tuna nomino ya kike katika sentensi. Tunapokumbana zaidi na maneno *na Tom*, tutainua bendera inayosema tuna nomino ya wingi. Hivyo kwa kudhibiti hali tunazidi kufuatilia sifa za sarufi za sentensi.

> **Kumbuka**: Hapa kuna rasilimali nzuri ya kuelewa mambo ya ndani ya LSTM: [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) na Christopher Olah.

Ingawa muundo wa ndani wa seli ya LSTM unaweza kuonekana mgumu, Keras huficha utekelezaji huu ndani ya tabaka la `LSTM`, hivyo kitu pekee tunachohitaji kufanya katika mfano hapo juu ni kubadilisha tabaka la marudio:


In [8]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

15000/15000 [==============================] - 188s 13ms/step - loss: 0.5692 - acc: 0.7916 - val_loss: 0.3441 - val_acc: 0.8870


> **Kumbuka** kwamba kufundisha LSTMs pia ni polepole sana, na huenda usione ongezeko kubwa la usahihi mwanzoni mwa mafunzo. Huenda ukahitaji kuendelea kufundisha kwa muda fulani ili kufanikisha usahihi mzuri.


## RNNs za mwelekeo miwili na tabaka nyingi  

Katika mifano yetu hadi sasa, mitandao inayorudiarudia hufanya kazi kuanzia mwanzoni mwa mtiririko hadi mwisho. Hii inahisi ya asili kwetu kwa sababu inafuata mwelekeo uleule tunaosoma au kusikiliza hotuba. Hata hivyo, kwa hali zinazohitaji upatikanaji wa nasibu wa mfuatano wa input, ni mantiki zaidi kuendesha hesabu ya kurudiarudia katika mwelekeo yote miwili. RNNs zinazoruhusu hesabu katika mwelekeo yote miwili huitwa **bidirectional** RNNs, na zinaweza kuundwa kwa kufunika tabaka la kurudiarudia na tabaka maalum la `Bidirectonal`.  

> **Kumbuka**: Tabaka la `Bidirectional` hufanya nakala mbili za tabaka lililomo ndani yake, na linaweka mali ya `go_backwards` ya moja ya nakala hizo kuwa `True`, ikifanya iende katika mwelekeo wa kinyume katika mfuatano.  

Mitandao inayorudiarudia, iwe ya mwelekeo mmoja au mwelekeo miwili, inakamata mifumo ndani ya mfuatano, na kuihifadhi katika vector za hali au kuirudisha kama matokeo. Kama ilivyo kwa mitandao ya convolutional, tunaweza kujenga tabaka nyingine ya kurudiarudia ikifuata ile ya kwanza ili kukamata mifumo ya ngazi ya juu, iliyojengwa kutoka kwa mifumo ya ngazi ya chini iliyochimbwa na tabaka la kwanza. Hii inatupeleka kwenye dhana ya **multi-layer RNN**, ambayo ina mitandao miwili au zaidi inayorudiarudia, ambapo matokeo ya tabaka lililotangulia hupitishwa kwa tabaka linalofuata kama input.  

![Picha inaonyesha Multilayer long-short-term-memory- RNN](../../../../../translated_images/sw/multi-layer-lstm.dd975e29bb2a59fe.webp)  

*Picha kutoka [chapisho hili nzuri](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3) la Fernando López.*  

Keras hufanya uundaji wa mitandao hii kuwa rahisi, kwa sababu unahitaji tu kuongeza tabaka zaidi za kurudiarudia kwenye mfano. Kwa tabaka zote isipokuwa ya mwisho, tunahitaji kubainisha kipengele cha `return_sequences=True`, kwa sababu tunahitaji tabaka kurudisha hali zote za kati, sio tu hali ya mwisho ya hesabu ya kurudiarudia.  

Hebu tujenge LSTM ya tabaka mbili ya mwelekeo miwili kwa tatizo letu la uainishaji.  

> **Kumbuka** msimbo huu tena unachukua muda mrefu sana kumaliza, lakini unatupa usahihi wa juu kabisa tuliouona hadi sasa. Kwa hivyo labda inafaa kusubiri na kuona matokeo.  


In [9]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

5045/7500 [===================>..........] - ETA: 2:33 - loss: 0.3709 - acc: 0.8706

## RNNs kwa kazi nyingine

Hadi sasa, tumelenga kutumia RNNs kuchambua mfululizo wa maandishi. Lakini zinaweza kushughulikia kazi nyingi zaidi, kama vile uundaji wa maandishi na tafsiri ya mashine &mdash; tutazingatia kazi hizo katika kitengo kinachofuata.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Kionyozo**:
Hati hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kupata usahihi, tafadhali fahamu kwamba tafsiri za kiotomatiki zinaweza kuwa na makosa au upungufu wa usahihi. Hati ya asili katika lugha yake halisi inapaswa kuchukuliwa kama chanzo cha mamlaka. Kwa taarifa muhimu, tafsiri ya kitaalamu inayofanywa na binadamu inapendekezwa. Hatutojibu kwa kuelewa vibaya au tafsiri potofu zinazotokea kutokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
